# Step by Step Guide to run models on Raspberry Pi Zero 2W

- Import models
- Create a Virtual environment
- Activate it
- pip install jupyter
- run jupyter notebook --ip=[YOUR IP ADDREES] --no-browser
- Open on Browser or VSCode with link

## Teste Microfone & Python:

In [1]:
!pwd

/home/rolds/Documents/KEYWORD_SPOTTING


In [2]:
!arecord -l

**** List of CAPTURE Hardware Devices ****
card 1: SoloCast [HyperX SoloCast], device 0: USB Audio [USB Audio]
  Subdevices: 1/1
  Subdevice #0: subdevice #0


In [3]:
!ls

keyw_env  models  output.wav


In [4]:
import pyaudio

p = pyaudio.PyAudio()
info = p.get_host_api_info_by_index(0)
numdevices = info.get('deviceCount')

print("Dispositivos de áudio encontrados:")
for i in range(0, numdevices):
    device_info = p.get_device_info_by_host_api_device_index(0, i)
    if device_info.get('maxInputChannels') > 0:
        print(f"  Índice de Entrada {i}: {device_info.get('name')}")

p.terminate()

Dispositivos de áudio encontrados:
  Índice de Entrada 0: HyperX SoloCast: USB Audio (hw:1,0)


ALSA lib pcm_asym.c:105:(_snd_pcm_asym_open) capture slave is not defined
ALSA lib confmisc.c:1369:(snd_func_refer) Unable to find definition 'cards.0.pcm.front.0:CARD=0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5703:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM front
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib confmisc.c:1369:(snd_func_refer) Unable to find definition 'cards.0.pcm.surround51.0:CARD=0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5703:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM surr

In [5]:
import pyaudio
import wave
import numpy as np
from scipy.signal import resample_poly

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
CHUNK = 1024
RECORD_SECONDS = 10
DEVICE_INDEX = 0   # replace this with your detected USB mic's index
WAVE_OUTPUT_FILENAME = "output.wav"

TARGET_RATE = 16000

audio = pyaudio.PyAudio()

stream = audio.open(format=FORMAT, channels=CHANNELS,
                    rate=RATE, input=True, input_device_index=DEVICE_INDEX,
                    frames_per_buffer=CHUNK)

print("Recording...")
frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Finished recording.")

stream.stop_stream()
stream.close()
audio.terminate()

audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# Calcula os fatores para a reamostragem (de 48k para 16k é 1/3)
resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

# Converte de volta para int16
resampled_data = resampled_data.astype(np.int16)

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(RATE)
wf.writeframes(b''.join(frames))
wf.close()

ALSA lib pcm_asym.c:105:(_snd_pcm_asym_open) capture slave is not defined
ALSA lib confmisc.c:1369:(snd_func_refer) Unable to find definition 'cards.0.pcm.front.0:CARD=0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5703:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM front
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib confmisc.c:1369:(snd_func_refer) Unable to find definition 'cards.0.pcm.surround51.0:CARD=0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5703:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM surr

Recording...
Finished recording.


In [6]:
import IPython.display as ipd
ipd.Audio("output.wav")

## Teste Modelo:

In [12]:
import time
import numpy as np
import pyaudio
import librosa
import tflite_runtime.interpreter as tflite
from collections import deque
from scipy.signal import resample_poly
from gpiozero import LED
import threading
import queue

CLASS_NAMES = ['_silence_', '_unknown_', 'go', 'no', 'off', 'on', 'stop']

MODEL_PATH = "./models/model_mfcc_fp16.tflite"

NATIVE_RATE = 48000
TARGET_RATE = 16000
AUDIO_DURATION_S = 1
SAMPLES_PER_AUDIO_TARGET = TARGET_RATE * AUDIO_DURATION_S
SAMPLES_PER_AUDIO_NATIVE = NATIVE_RATE * AUDIO_DURATION_S

CHUNK = 1024
DEVICE_INDEX = 0
CHANNELS = 1
FORMAT = pyaudio.paInt16

N_FFT = 512
FRAME_STEP = 256
N_MELS = 128
N_MFCCS = 40
F_MIN = 20
F_MAX = TARGET_RATE / 2

print("Carregando modelo TFLite...")
interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Modelo carregado com sucesso.")
print("Input shape:", input_details[0]['shape'])

# --- 2. FUNÇÃO DE PRÉ-PROCESSAMENTO (CORRIGIDA) ---
def preprocess_audio(audio_data):
    """
    Recebe 1 segundo de áudio, extrai os MFCCs e formata
    para a entrada do modelo.
    """
    mfccs = librosa.feature.mfcc(
        y=audio_data, sr=TARGET_RATE, n_mfcc=N_MFCCS, n_fft=N_FFT,
        hop_length=FRAME_STEP, win_length=N_FFT, n_mels=N_MELS,
        fmin=F_MIN, fmax=F_MAX, center=False
    )
    
    # --- LINHA ADICIONADA AQUI ---
    # Transpõe a matriz para (tempo, features) -> (61, 40)
    mfccs = mfccs.T
    
    # Adiciona a dimensão do canal (necessário para modelos CNN)
    mfccs = np.expand_dims(mfccs, axis=-1)
    
    # Adiciona a dimensão do batch (o modelo espera um lote de dados)
    mfccs = np.expand_dims(mfccs, axis=0)
    
    return mfccs.astype(np.float32)

# --- 3. LOOP DE INFERÊNCIA EM TEMPO REAL ---
audio = pyaudio.PyAudio()
stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=NATIVE_RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

audio_buffer = deque(maxlen=SAMPLES_PER_AUDIO_NATIVE)

inference_trigger_count = 40 # Dispara a cada ~0.5 segundos (20 * 1024 / 48000)
chunk_counter = 0

print("\n--- Pressione CTRL+C para parar ---")
print("Ouvindo...")

try:
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)
        chunk_data = np.frombuffer(data, dtype=np.int16)
        audio_buffer.extend(chunk_data)
        chunk_counter += 1

        # Gatilho de inferência: executa quando o contador atinge o limite E o buffer está cheio
        if chunk_counter >= inference_trigger_count and len(audio_buffer) == SAMPLES_PER_AUDIO_NATIVE:
            # Pega o snapshot atual do buffer (1 segundo de áudio a 48kHz)
            waveform_native = np.array(audio_buffer, dtype=np.float32)
            
            # --- REAMOSTRAGEM (RESAMPLING) ---
            waveform_resampled = resample_poly(waveform_native, TARGET_RATE, NATIVE_RATE)
            waveform_resampled = waveform_resampled.astype(np.float32) # Garante float32
            
            # Normaliza o áudio para o intervalo [-1, 1]
            waveform_final = waveform_resampled / 32768.0

            # Realiza o pré-processamento
            input_data = preprocess_audio(waveform_final)

            # Executa a inferência
            interpreter.set_tensor(input_details[0]['index'], input_data)
            interpreter.invoke()
            
            output_data = interpreter.get_tensor(output_details[0]['index'])
            prediction_index = np.argmax(output_data[0])
            predicted_class = CLASS_NAMES[prediction_index]
            confidence = np.max(output_data[0])

            if predicted_class != '_unknown_' and confidence > 0.9:
                print(f"DETECÇÃO: {predicted_class} (Confiança: {confidence:.2f})")
            else:
                pass
            
            # Reseta o contador
            chunk_counter = 0

except KeyboardInterrupt:
    print("\nParando a execução...")

finally:
    stream.stop_stream()
    stream.close()
    audio.terminate()
    print("Recursos liberados.")


Carregando modelo TFLite...
Modelo carregado com sucesso.
Input shape: [ 1 61 40  1]


ALSA lib pcm_asym.c:105:(_snd_pcm_asym_open) capture slave is not defined
ALSA lib confmisc.c:1369:(snd_func_refer) Unable to find definition 'cards.0.pcm.front.0:CARD=0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5703:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM front
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib confmisc.c:1369:(snd_func_refer) Unable to find definition 'cards.0.pcm.surround51.0:CARD=0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5703:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2666:(snd_pcm_open_noupdate) Unknown PCM surr


--- Pressione CTRL+C para parar ---
Ouvindo...
DETECÇÃO: stop (Confiança: 1.00)
DETECÇÃO: no (Confiança: 1.00)
DETECÇÃO: no (Confiança: 1.00)
DETECÇÃO: on (Confiança: 0.99)
DETECÇÃO: off (Confiança: 1.00)
DETECÇÃO: stop (Confiança: 1.00)
DETECÇÃO: off (Confiança: 1.00)
DETECÇÃO: off (Confiança: 1.00)

Parando a execução...
Recursos liberados.


## Teste LEDs:

In [13]:
import time
import numpy as np
import pyaudio
import librosa
import tflite_runtime.interpreter as tflite
from collections import deque
from scipy.signal import resample_poly
from gpiozero import LED
import threading
import queue

In [14]:
CLASS_NAMES = ['_silence_', '_unknown_', 'go', 'no', 'off', 'on', 'stop']

MODEL_PATH = "./models/model_mfcc_fp16.tflite"

NATIVE_RATE = 48000
TARGET_RATE = 16000
AUDIO_DURATION_S = 1
SAMPLES_PER_AUDIO_TARGET = TARGET_RATE * AUDIO_DURATION_S
SAMPLES_PER_AUDIO_NATIVE = NATIVE_RATE * AUDIO_DURATION_S

CHUNK = 1024
DEVICE_INDEX = 0
CHANNELS = 1
FORMAT = pyaudio.paInt16

N_FFT = 512
FRAME_STEP = 256
N_MELS = 128
N_MFCCS = 40
F_MIN = 20
F_MAX = TARGET_RATE / 2

In [15]:
def inference_thread_func(prediction_queue):
    """
    Esta função roda em sua própria thread. Ela captura e processa o áudio
    continuamente e coloca as previsões em uma 'queue' para a outra thread.
    """
    print("[Thread Inferência] Carregando modelo TFLite...")
    interpreter = tflite.Interpreter(model_path=MODEL_PATH)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    print("[Thread Inferência] Modelo carregado.")

    audio = pyaudio.PyAudio()
    stream = audio.open(format=FORMAT, channels=CHANNELS, rate=NATIVE_RATE, input=True,
                        input_device_index=DEVICE_INDEX, frames_per_buffer=CHUNK)
    
    audio_buffer = deque(maxlen=SAMPLES_PER_AUDIO_NATIVE)
    inference_trigger_count = 20
    chunk_counter = 0

    print("[Thread Inferência] Ouvindo...")
    try:
        while True:
            data = stream.read(CHUNK, exception_on_overflow=False)
            chunk_data = np.frombuffer(data, dtype=np.int16)
            audio_buffer.extend(chunk_data)
            chunk_counter += 1

            if chunk_counter >= inference_trigger_count and len(audio_buffer) == SAMPLES_PER_AUDIO_NATIVE:
                waveform_native = np.array(audio_buffer, dtype=np.float32)
                waveform_resampled = resample_poly(waveform_native, TARGET_RATE, NATIVE_RATE).astype(np.float32)
                waveform_final = waveform_resampled / 32768.0

                mfccs = librosa.feature.mfcc(y=waveform_final, sr=TARGET_RATE, n_mfcc=N_MFCCS, n_fft=N_FFT,
                                             hop_length=FRAME_STEP, n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, center=False)
                mfccs = mfccs.T
                if mfccs.shape[0] > 61: mfccs = mfccs[:61, :]
                elif mfccs.shape[0] < 61: mfccs = np.pad(mfccs, ((0, 61 - mfccs.shape[0]), (0, 0)), mode='constant')
                
                input_data = np.expand_dims(mfccs, axis=-1)
                input_data = np.expand_dims(input_data, axis=0).astype(np.float32)

                interpreter.set_tensor(input_details[0]['index'], input_data)
                interpreter.invoke()
                
                output_data = interpreter.get_tensor(output_details[0]['index'])
                prediction_index = np.argmax(output_data[0])
                predicted_class = CLASS_NAMES[prediction_index]
                confidence = np.max(output_data[0])
                
                prediction_queue.put({'class': predicted_class, 'confidence': confidence})
                chunk_counter = 0
    finally:
        print("[Thread Inferência] Encerrando...")
        stream.stop_stream()
        stream.close()
        audio.terminate()

In [16]:
def gpio_thread_func(prediction_queue):
    """
    Esta função roda em sua própria thread. Ela usa gpiozero para controlar os LEDs
    com base nas previsões recebidas da 'queue'.
    """
    print("[Thread GPIO] Configurando dispositivos com gpiozero...")
    
    # Cria objetos LED para cada pino. A configuração é automática.
    led_vermelho = LED(4)
    led_azul = LED(5)
    
    print("[Thread GPIO] Dispositivos configurados. Aguardando previsões...")
    
    # O bloco try/except aqui é apenas para a mensagem de encerramento.
    # gpiozero cuida da limpeza dos pinos automaticamente.
    try:
        while True:
            # Pega uma previsão da "caixa de correio".
            prediction = prediction_queue.get()
            
            p_class = prediction['class']
            p_conf = prediction['confidence']

            #print(f"[Thread GPIO] Recebido: Classe={p_class}, Confiança={p_conf:.2f}")

            if p_conf > 0.99:
                if p_class == 'go':
                    led_vermelho.on()
                    led_azul.off()
                    print("[Thread GPIO] AÇÃO: LED LIGAR ACESO")
                elif p_class == 'stop':
                    led_azul.on()
                    led_vermelho.off()
                    print("[Thread GPIO] AÇÃO: LED DESLIGAR ACESO")
                
                elif p_class == 'no':
                    raise KeyboardInterrupt
                
                elif p_class == 'off':
                    led_vermelho.off()
                    led_azul.off()
                    print("[Thread GPIO] AÇÃO: TODOS OS LEDS APAGADOS")
                    
                elif p_class == 'on':
                    led_vermelho.on()
                    led_azul.on()
                    print("[Thread GPIO] AÇÃO: TODOS OS LEDS ACESOS")


    except (KeyboardInterrupt, SystemExit):
        print("[Thread GPIO] Encerrando...")
        led_vermelho.close()
        led_azul.close()
        print("[Thread GPIO] Dispositivos liberados.")

In [18]:
prediction_queue = queue.Queue()

thread1 = threading.Thread(target=inference_thread_func, args=(prediction_queue,), daemon=True)
thread2 = threading.Thread(target=gpio_thread_func, args=(prediction_queue,), daemon=True)

print("[Main] Iniciando as threads...")
thread1.start()
thread2.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n[Main] Programa encerrado pelo usuário.")

Exception in thread Thread-11 (gpio_thread_func):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/home/rolds/Documents/KEYWORD_SPOTTING/keyw_env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_28293/2414308407.py", line 9, in gpio_thread_func
  File "/home/rolds/Documents/KEYWORD_SPOTTING/keyw_env/lib/python3.11/site-packages/gpiozero/devices.py", line 108, in __call__
    self = super().__call__(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/rolds/Documents/KEYWORD_SPOTTING/keyw_env/lib/python3.11/site-packages/gpiozero/output_devices.py", line 192, in __init__
    super().__init__(pin, active_high=active_high,
  File "/home/rolds/Documents/KEYWORD_SPOTTING/keyw_env/lib/python3.11/site

[Main] Iniciando as threads...
[Thread Inferência] Carregando modelo TFLite...
[Thread Inferência] Modelo carregado.
[Thread GPIO] Configurando dispositivos com gpiozero...


Exception in thread Thread-10 (inference_thread_func):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
Expression 'ret' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 1736
Expression 'AlsaOpen( &alsaApi->baseHostApiRep, params, streamDir, &self->pcm )' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 1904
Expression 'PaAlsaStreamComponent_Initialize( &self->capture, alsaApi, inParams, StreamDirection_In, NULL != callback )' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 2171
Expression 'PaAlsaStream_Initialize( stream, alsaHostApi, inputParameters, outputParameters, sampleRate, framesPerBuffer, callback, streamFlags, userData )' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 2839
    self.run()
  File "/home/rolds/Documents/KEYWORD_SPOTTING/keyw_env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 975


[Main] Programa encerrado pelo usuário.
